# NS06 Tutorial B - Katz Centrality

**Prerequisites:** eigenvector centrality, directed graphs, and basic matrix notation.

**Learning goals**
- Understand why Katz centrality adds a **baseline** term.
- Compute Katz centrality from the fixed-point equation
  $$\mathbf{x} = \alpha A^T \mathbf{x} + \beta \mathbf{1}$$
- Compare iterative, matrix-based, and NetworkX computations.
- Interpret how `alpha` and `beta` change **status/prestige** rankings.
- Connect Katz centrality to directed real-world networks such as citations.

**Outline**
1. Citation-network motivation.
2. Katz vs eigenvector centrality on a DAG.
3. Iterative and closed-form computation.
4. Choosing and interpreting `alpha`.
5. Exercise and takeaway.

In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()

## 1. Citation-network motivation

In the lecture, Katz centrality is introduced as a measure of **status/prestige** on directed networks.

A citation network is the natural computational example:

- a paper gains prestige if it is cited by important papers
- but a paper with zero incoming citations should not be forced to zero forever

The baseline parameter $\beta$ fixes exactly this problem.

In [ ]:
citation = nx.DiGraph([
    ('IntroML', 'DeepVision'),
    ('IntroML', 'GraphMiner'),
    ('LinearModels', 'GraphMiner'),
    ('DeepVision', 'TrustworthyAI'),
    ('GraphMiner', 'TrustworthyAI'),
    ('DataEthics', 'TrustworthyAI'),
])

pos_citation = {
    'IntroML': (0, 1),
    'LinearModels': (0, 0),
    'DataEthics': (0, -1),
    'DeepVision': (1, 0.8),
    'GraphMiner': (1, -0.2),
    'TrustworthyAI': (2, 0.3),
}

plot_graph(
    citation,
    title='Toy citation network',
    pos=pos_citation,
    with_labels=True,
    node_size=2000,
    arrows=True,
    arrowsize=18,
    font_size=9,
)

print('Raw in-degree (citation count):')
print(pd.Series(dict(citation.in_degree())).sort_values(ascending=False).to_string())

## 2. Katz versus eigenvector centrality on the same DAG

On a DAG, pure eigenvector centrality collapses toward the terminal nodes.

Katz keeps the recursive idea but adds a baseline term, so source nodes retain non-zero status.

In [ ]:
alpha = 0.10
beta = 1.0

eig_citation = nx.eigenvector_centrality(citation, max_iter=1000)
katz_citation = nx.katz_centrality(citation, alpha=alpha, beta=beta)

comparison = pd.DataFrame({
    'node': list(citation.nodes()),
    'in_degree': [citation.in_degree(n) for n in citation.nodes()],
    'eigenvector': [eig_citation[n] for n in citation.nodes()],
    'katz': [katz_citation[n] for n in citation.nodes()],
}).sort_values('katz', ascending=False)

print(comparison.round(6).to_string(index=False))

## 3. Iterative and closed-form computation

For a graph stored as `i -> j`, prestige flows through **incoming** links, so the relevant matrix is $A^T$.

Two computational views are useful:

- **fixed-point iteration**: repeated updates until convergence
- **closed form**: solve $(I - \alpha A^T)\mathbf{x} = \beta \mathbf{1}$ directly

In [ ]:
def katz_from_scratch(G, alpha, beta=1.0, nodelist=None, max_iter=200, tol=1e-12):
    if nodelist is None:
        nodelist = list(G.nodes())
    influence = nx.to_numpy_array(G, nodelist=nodelist, dtype=float).T
    x = np.ones(len(nodelist), dtype=float)
    history = []

    for iteration in range(max_iter):
        x_new = alpha * (influence @ x) + beta * np.ones(len(nodelist))
        delta = float(np.linalg.norm(x_new - x))
        history.append({'iteration': iteration + 1, 'delta': delta, **{node: value for node, value in zip(nodelist, x_new)}})
        if delta < tol:
            x = x_new
            break
        x = x_new

    x_norm = x / np.linalg.norm(x)
    return dict(zip(nodelist, x_norm)), pd.DataFrame(history)


def katz_closed_form(G, alpha, beta=1.0, nodelist=None):
    if nodelist is None:
        nodelist = list(G.nodes())
    influence = nx.to_numpy_array(G, nodelist=nodelist, dtype=float).T
    I = np.eye(len(nodelist))
    x = np.linalg.solve(I - alpha * influence, beta * np.ones(len(nodelist)))
    x_norm = x / np.linalg.norm(x)
    return dict(zip(nodelist, x_norm))

order = list(citation.nodes())
katz_manual, katz_history = katz_from_scratch(citation, alpha=alpha, beta=beta, nodelist=order)
katz_direct = katz_closed_form(citation, alpha=alpha, beta=beta, nodelist=order)

compare_manual = pd.DataFrame({
    'node': order,
    'iteration': [katz_manual[n] for n in order],
    'closed_form': [katz_direct[n] for n in order],
    'networkx': [katz_citation[n] for n in order],
})
compare_manual['max_diff'] = compare_manual[['iteration', 'closed_form', 'networkx']].max(axis=1) - compare_manual[['iteration', 'closed_form', 'networkx']].min(axis=1)

print('First iterations of the Katz update:')
print(katz_history.head(6).round(6).to_string(index=False))
print('\nIterative, closed-form, and NetworkX results:')
print(compare_manual.round(6).to_string(index=False))

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.plot(katz_history['iteration'], katz_history['delta'], marker='o', color=NODE_COLOR, linewidth=2)
style_axis(ax, title='Convergence of the Katz fixed-point iteration', xlabel='iteration', ylabel='||x^(t+1) - x^(t)||', yscale='log')
plt.show()

This is the core computational message:

- iteration shows the fixed-point logic directly
- the matrix solve shows the linear-algebra structure directly
- NetworkX gives the same ranking efficiently

These are three views of the same object.

## 4. Choosing and interpreting $\alpha$

The lecture condition is

$$
\alpha < \frac{1}{\lambda_{\max}}
$$

where $\lambda_{\max}$ is the largest eigenvalue of the influence matrix.

On a DAG, the spectral radius is zero, so the usual bound is vacuous. The practical interpretation remains the same: larger $\alpha$ gives more weight to longer walks.

In [ ]:
influence = nx.to_numpy_array(citation, nodelist=order, dtype=float).T
spectral_radius = max(abs(np.linalg.eigvals(influence)))
print(f'spectral radius rho(A^T) = {spectral_radius:.6f}')
if spectral_radius == 0:
    print('This graph is acyclic, so the usual convergence bound does not constrain alpha.')
    print('Alpha is still a modeling choice: larger alpha gives more weight to longer walks.')
else:
    print(f'A safe choice is alpha < {1 / spectral_radius:.6f}.')

alpha_grid = np.linspace(0.05, 0.60, 12)
tracked_nodes = ['IntroML', 'GraphMiner', 'TrustworthyAI']
alpha_rows = []
for alpha_test in alpha_grid:
    scores = katz_closed_form(citation, alpha=alpha_test, beta=1.0, nodelist=order)
    row = {'alpha': alpha_test}
    for node in tracked_nodes:
        row[node] = scores[node]
    alpha_rows.append(row)
alpha_df = pd.DataFrame(alpha_rows)

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
for node, color in zip(tracked_nodes, [CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['green']]):
    ax.plot(alpha_df['alpha'], alpha_df[node], marker='o', linewidth=2, color=color, label=node)
style_axis(ax, title='Effect of alpha on Katz prestige', xlabel='alpha', ylabel='normalized Katz score', legend=True)
plt.show()

**Real-world implication.**

On a citation network, larger $\alpha$ means that indirect chains of influence matter more.

A paper can then gain prestige not only from its direct citations, but from how influential those citing papers are inside the larger citation cascade.

In [ ]:
plot_metric(
    citation,
    katz_citation,
    title='Citation network: size / color = Katz centrality',
    pos=pos_citation,
    with_labels=True,
    min_node_size_px=26,
    max_node_size_px=60,
    arrows=True,
    arrowsize=18,
    font_size=9,
)

## 5. Exercise

Keep $\alpha = 0.10$ and increase $\beta$ from `1.0` to `2.0`.

Before running the next cell, predict:

- which nodes benefit the most from a larger baseline?
- does the ranking change, or mainly the gaps?

In [ ]:
for beta_test in [1.0, 2.0]:
    scores = nx.katz_centrality(citation, alpha=0.10, beta=beta_test)
    table = pd.DataFrame({
        'node': list(citation.nodes()),
        'katz': [scores[n] for n in citation.nodes()],
    }).sort_values('katz', ascending=False)
    print(f'beta = {beta_test:.1f}')
    print(table.round(6).to_string(index=False))
    print()

## Takeaway

- **Katz centrality** extends eigenvector centrality with the baseline term $\beta$.
- It is the natural prestige/status measure when pure recursive support collapses on directed graphs.
- Computationally, you can read it as an iteration, a linear solve, or a NetworkX routine.
- The parameter $\alpha$ controls the weight of longer walks, and $\beta$ keeps source nodes alive in the ranking.